# Retrieval-Augmented Generation (RAG) with Ollama

## Overview

In this notebook, you'll build a simple Retrieval-Augmented Generation (RAG) system that answers questions using the contents of a PDF document.

### Learning Objectives

- Read and extract text from a PDF.
- Split text into meaningful chunks.
- Build a TF-IDF vector index.
- Retrieve the most relevant chunks.
- Use a local LLM via Ollama to answer questions.
- Restrict the model to answering only from the provided context.

### Architecture

```
                ┌───────────────┐
                │   PDF File    │
                └──────┬────────┘
                       │
                Extract Text
                       │
                Chunk Document
                       │
                 TF-IDF Index
                       │
                 User Question
                       │
              Similarity Search
                       │
                Relevant Chunks
                       │
                    Ollama
                       │
                 Final Answer
```

In [ ]:
# Standard Library
from pathlib import Path

# PDF Reader
from pypdf import PdfReader

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Numerical Computing
import numpy as np

# Ollama Client
import ollama

## Step 1 — Load the PDF

We'll load a PDF document and extract all of its text into a single string.

In [ ]:
PDF_PATH = Path("assets/claude_certification_foundation_associate.pdf")

reader = PdfReader(PDF_PATH)

print(f"Pages: {len(reader.pages)}")

In [ ]:
document_text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        document_text += page_text + "\n"

print(f"Characters extracted: {len(document_text):,}")

## Step 2 — Split the Document into Chunks

Large language models cannot efficiently process entire documents at once.

Instead, we'll split the document into overlapping chunks so that relevant context can be retrieved later.

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    """
    Split text into overlapping chunks.
    """
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

In [ ]:
chunks = chunk_text(document_text)

print(f"Total Chunks: {len(chunks)}")

In [ ]:
chunks[0][:500]

## Step 3 — Build a TF-IDF Vector Index

TF-IDF converts each chunk into a numerical vector based on word importance.

These vectors allow us to retrieve the chunks most relevant to a user's question using cosine similarity.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")

chunk_vectors = vectorizer.fit_transform(chunks)

print(chunk_vectors.shape)

## Step 4 — Create a Retriever

The retriever compares the user's question to every chunk in the document and returns the most similar chunks.

In [ ]:
def retrieve(query, top_k=3):
    """
    Retrieve the top-k most relevant chunks for a given query.
    """
    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(query_vector, chunk_vectors).flatten()

    indices = np.argsort(similarities)[::-1][:top_k]

    return [chunks[i] for i in indices]

In [ ]:
question = "What topics are covered in the certification?"

results = retrieve(question)

for idx, chunk in enumerate(results, start=1):
    print("=" * 80)
    print(f"Chunk {idx}")
    print("=" * 80)
    print(chunk[:500])

## Step 5 — Query Ollama

We'll provide the retrieved context to a local language model and instruct it to answer **only** using the supplied information.

In [ ]:
SYSTEM_PROMPT = """
You are a helpful AI assistant.

Answer the user's question ONLY using the supplied context.

If the answer is not present in the context, reply:

"I couldn't find the answer in the provided document."

Do not invent information.
"""

In [ ]:
def ask_rag(question):
    context = "\n\n".join(retrieve(question))

    prompt = f"""
Context:

{context}

Question:

{question}
"""

    response = ollama.chat(
        model="qwen3",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )

    return response["message"]["content"]

In [ ]:
question = "Summarize the certification in five bullet points."

answer = ask_rag(question)

print(answer)

## Exercises

Try asking different questions, such as:

- What are the prerequisites?
- Which skills are covered?
- How is the exam structured?
- What are the recommended study areas?

Experiment with:

- Different chunk sizes.
- Different overlap values.
- Increasing or decreasing `top_k`.
- Replacing TF-IDF with semantic embeddings (e.g., SentenceTransformers or FAISS) for improved retrieval quality.